<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Gradient descent

**[Gradient descent](https://learning.nextia-ai.com/courses/deep-learning/m03/gradient-descent/)** · Deep Learning and Transformers Explained · Module 3, lesson 2 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the bottom of a loss curve with small downhill steps: measure the slope, take a step of learning rate × slope, and repeat, first for one weight and then for two.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account, no graphics card. |
| **You should know** | The loss, −ln p of the right team, from [Loss: how wrong is the network?](https://learning.nextia-ai.com/courses/deep-learning/m03/loss/). The slope of a curve, from [Loss and gradients](https://learning.nextia-ai.com/courses/math/m04/loss-and-gradients/) in the mathematics course. |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/gradient-descent/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M03-L02-gradient-descent/gradient-descent-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: the tickets as numbers

The next cell turns the tickets into word counts, as in [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/): a vocabulary from the training tickets, then one row of counts per ticket. It also makes `loss_fn`, the loss of this module. Run it.

> **Check.** You should see `x_train shape (4696, 2864)`, `x_valid shape (600, 2864)` and `vocabulary: 2864`.

In [ ]:
import torch
from torch import nn

# The tickets as numbers: a vocabulary from the training tickets, then word counts.
vocab = Vocab([r["text"] for r in train_rows])
x_train, y_train = bag_of_words(train_rows, vocab), labels(train_rows)
x_valid, y_valid = bag_of_words(valid_rows, vocab), labels(valid_rows)
loss_fn = nn.CrossEntropyLoss()
show_shape("x_train", x_train)
show_shape("x_valid", x_valid)
print("vocabulary:", len(vocab), "| teams:", TEAMS)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A network with one weight

Four made-up tickets, and a network with one neuron. It answers one question: "Is this ticket for the payment team?" Its only input is x, how many times the ticket says "refund":

| Ticket | Text | x | Payment? |
|---|---|---|---|
| A | My parcel has not arrived yet. | 0 | no (delivery) |
| B | You confirmed my refund, but the money is not on my card. | 1 | yes |
| C | You confirmed my refund two weeks ago. Where is the refund? | 2 | yes |
| D | I sent the lamp back. Can I get a refund, or is there no refund on sale items? | 2 | no (returns) |

The neuron computes p = sigmoid(w·x + b): the probability of "payment". The bias b stays at −1 for now, so the network has **one weight, w**, to learn. The loss is −ln p for a payment ticket and −ln(1 − p) for the others, as in the last lesson; `loss(w)` returns the mean over the four tickets. The code uses plain Python, so you can follow every number.

> **Predict.** At w = 0, every ticket gets p = sigmoid(−1) = 0.269. Is the loss of the payment tickets or of the other tickets larger? Then run the cell.

In [ ]:
import math

refunds = [0, 1, 2, 2]      # how many times each ticket says "refund"
is_payment = [0, 1, 1, 0]   # 1 = the payment team, 0 = another team

def loss(w, b=-1.0):
    """The mean loss of the one-neuron network on the four tickets."""
    total = 0.0
    for x, y in zip(refunds, is_payment):
        p = 1 / (1 + math.exp(-(w * x + b)))       # the neuron: sigmoid(w·x + b)
        total += -math.log(p) if y == 1 else -math.log(1 - p)
    return total / len(refunds)

for w in [-1, 0, 0.5, 1, 2, 3]:
    print(f"w = {w:4}   loss = {loss(w):.4f}")

> **Check.** You should see six lines: `w =   -1   loss = 1.3843`, then 0.8133 at w = 0, 0.6684 at 0.5, 0.6582 at 1, 0.9309 at 2 and 1.3634 at 3.

At w = 0, tickets A and D cost −ln(1 − 0.269) = 0.313 each, and B and C cost −ln 0.269 = 1.313 each: the mean is 0.813. The loss falls from w = −1 to somewhere between 0.5 and 1, then rises again. The bottom is in between. Run the next cell to see the curve.

In [ ]:
import matplotlib.pyplot as plt

ws = [i / 20 for i in range(-30, 101)]          # w from -1.5 to 5.0
plt.figure(figsize=(7, 3))
plt.plot(ws, [loss(w) for w in ws])
plt.xlabel("weight w")
plt.ylabel("loss on the four tickets")
plt.show()

> **Check.** You should see a valley: high on the left, lowest a little below w = 1, and rising to the right.

## 2. The slope: which way is downhill?

You cannot see the whole curve for a real network. You can only feel the ground under your feet: the **slope** at the current w. Nudge w a tiny bit to each side and see how the loss changes. The slope is the change in loss divided by the change in w.

> **Predict.** At w = 0, is the slope negative (downhill to the right) or positive? And at w = 2? Then run the cell.

In [ ]:
def slope(w, nudge=1e-6):
    """The slope of the loss at w: nudge w both ways and compare."""
    return (loss(w + nudge) - loss(w - nudge)) / (2 * nudge)

print("slope at w = 0:", round(slope(0), 4))
print("slope at w = 2:", round(slope(2), 4))

> **Check.** You should see `slope at w = 0: -0.4138` and `slope at w = 2: 0.3853`.

At w = 0, the slope is negative: the loss falls when w grows, so downhill is to the **right**. At w = 2, the slope is positive: downhill is to the **left**. So always move **against** the slope.

## 3. Gradient descent by hand

The update rule: **new w = w − learning rate × slope**. The **learning rate** sets the size of the step. A steep slope gives a big step, a flat slope a small one.

> **Predict.** Start at w = 0 with a learning rate of 1. After three steps, will w be above or below 0.7? Then run the cell.

In [ ]:
w, rate = 0.0, 1.0
for step in range(4):
    print(f"step {step}   w = {w:.4f}   loss = {loss(w):.4f}   slope = {slope(w):.4f}")
    w = w - rate * slope(w)

> **Check.** You should see:
>
> ```
> step 0   w = 0.0000   loss = 0.8133   slope = -0.4138
> step 1   w = 0.4138   loss = 0.6839   slope = -0.2036
> step 2   w = 0.6174   loss = 0.6540   slope = -0.0902
> step 3   w = 0.7076   loss = 0.6481   slope = -0.0408
> ```

Each step goes downhill, and each step is smaller than the last, because the slope gets flatter near the bottom. The bottom of this curve is at about w = 0.78, with a loss of 0.6465.

## 4. Change one thing: the learning rate

> **Predict.** What happens with a learning rate 10 times larger, 10? Write your prediction, then run the cell.

In [ ]:
w, rate = 0.0, 10.0
for step in range(6):
    print(f"step {step}   w = {w:7.4f}   loss = {loss(w):.4f}")
    w = w - rate * slope(w)

> **Check.** You should see w jump from 0 to 4.1382 (loss 1.9084), then to −0.7510, then to 5.6220 (loss 2.6418), then to 0.6467, then to 1.3870.

With a learning rate of 10, every step jumps far past the bottom, to the other side, and the loss gets **worse** than at the start. The steps never settle. A learning rate that is too small has the opposite problem: with 0.1, the same three steps only reach w = 0.12.

## 5. Two parameters: the weight and the bias

Now let the bias b learn too. The loss is now a surface over two numbers, w and b, like a landscape. Each parameter gets its own slope: nudge one, keep the other fixed. The update rule is the same for both.

> **Predict.** Start at w = −1, b = 1, with a learning rate of 1. Will 60 steps reach the lowest point (loss 0.6465)? Then run the cell.

In [ ]:
def slope_w(w, b, nudge=1e-6):
    return (loss(w + nudge, b) - loss(w - nudge, b)) / (2 * nudge)

def slope_b(w, b, nudge=1e-6):
    return (loss(w, b + nudge) - loss(w, b - nudge)) / (2 * nudge)

w, b, rate = -1.0, 1.0, 1.0
for step in range(61):
    if step % 20 == 0:
        print(f"step {step:2}   w = {w:.4f}   b = {b:.4f}   loss = {loss(w, b):.4f}")
    w, b = w - rate * slope_w(w, b), b - rate * slope_b(w, b)

> **Check.** You should see the loss fall from 0.9082 at step 0 to 0.6656 at step 20, 0.6493 at step 40 and 0.6468 at step 60, with w = 0.7077 and b = −0.8727.

Close to the bottom, but not there yet: the valley is long and flat in one direction, so the steps become small. To search for the bottom without slopes, you would try many values of w and b. With 100 values each, that is 100 × 100 = 10,000 loss computations, for only two parameters.

## 6. The ticket network: every weight at once

The ticket network has 183,685 parameters. Gradient descent works the same way: one slope per parameter (together they are the **gradient**), and one step for all of them at once. The next cell runs five steps of plain gradient descent with a learning rate of 0.5 on all the training tickets. It uses `loss.backward()`, which computes all the slopes at once; the next lesson explains it, and you will write this loop yourself in [The training loop](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/).

> **Predict.** Will the loss fall at every step? Then run the cell.

In [ ]:
set_seed(0)
model = BagOfWordsNet(len(vocab))
print("parameters:", count_parameters(model))
for step in range(5):
    loss_value = loss_fn(model(x_train), y_train)
    model.zero_grad()
    loss_value.backward()                       # all 183,685 slopes (next lesson)
    with torch.no_grad():
        for weight in model.parameters():
            weight -= 0.5 * weight.grad         # new weight = weight - rate x slope
    print(f"step {step}   loss {loss_value.item():.4f}")

> **Check.** You should see `parameters: 183685`, then the loss `1.6236`, `1.5992`, `1.5781`, `1.5576` and `1.5376`.

All 183,685 parameters moved at once, and the loss fell at every step.

## 7. Failure case: a step uphill

Tomás writes the update with a plus sign: `w = w + rate * slope(w)`.

> **Predict.** What happens to the loss? Then run the cell.

In [ ]:
w, rate = 0.0, 1.0
for step in range(4):
    print(f"step {step}   w = {w:.4f}   loss = {loss(w):.4f}")
    w = w + rate * slope(w)          # wrong: this moves WITH the slope, uphill

> **Check.** You should see the loss grow at every step: 0.8133, 1.0177, 1.3685, 1.8370, while w goes further below 0.

A plus sign moves **with** the slope: uphill. The symptom is a loss that grows steadily, step after step, even with a small learning rate. **Fix:** subtract, `w - rate * slope`. PyTorch's optimizers subtract for you, which is one reason to use them.

## 8. Check yourself

> **Your turn.** Start at w = 0 with a learning rate of 0.3, and make **five** steps with the update rule. Put the final w in `w_after_5`, as a Python number. Then run the check cell.

In [ ]:
w_after_5 = ...  # five steps of gradient descent from w = 0, learning rate 0.3

In [ ]:
expect('w_after_5', w_after_5, 0.4616, tolerance=0.001)

## Next

You can now walk down a loss curve: measure the slope, step against it with learning rate × slope, and repeat, for one parameter or for many. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m03/gradient-descent/) has the two explorers (one weight, then the weight and the bias on a map) and the knowledge checks. The next lesson shows how PyTorch computes all the slopes at once, without nudging: backpropagation.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Backpropagation and autograd](https://learning.nextia-ai.com/courses/deep-learning/m03/backpropagation-and-autograd/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m03/gradient-descent/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets are made up for this course by a language model; no real customer wrote them.